In [ ]:
!pip uninstall cupy-cuda13x

In [ ]:
# Make sure your dependencies are installed
!pip install cupy-cuda12x gpu4pyscf-cuda12x cutensor-cu12 geometric basis_set_exchange pyscf-dispersion

import pyscf
from pyscf import gto, dft
from pyscf.geomopt.geometric_solver import optimize

# 1. Mount Google Drive if not already done in a previous cell
from google.colab import drive
drive.mount('/content/drive')

# 2. Set the path to the XYZ file generated earlier
file_path = "/content/drive/MyDrive/fluoro_Ax_TZV_XYZ.xyz"

# 3. Read the file, skipping the first 2 header lines (Atom Count & Comment)
with open(file_path, "r") as f:
    lines = f.readlines()[2:]
    cartesian_coordinates = "".join(lines)

# 4. Initialize PySCF Molecule using the specified Triple-Zeta Polarised Diffuse Basis Set
mol = gto.M(
    atom=cartesian_coordinates,
    basis='def2-tzvpd',  # Modern triple-zeta with diffuse and polarization for Sulfur
    charge=0,
    spin=0
)

# 5. Run geometry optimization using your GPU accelerated environment with CAM-B3LYP-D3(BJ)
mf = dft.RKS(mol).density_fit()
mf.xc = 'cam-b3lyp-d3bj'    # Activates CAM-B3LYP functional with Grimme's D3(BJ) dispersion
mf = mf.to_gpu()            # Transfers the functional, dispersion settings, and molecule to the GPU

mol_opt = optimize(mf)


In [ ]:
# =====================================================================
# BLOCK 1: EXTRACT AND SAVE OPTIMIZED GEOMETRY TO GOOGLE DRIVE
# =====================================================================

# 1. Get the final optimized geometry from the geomeTRIC output object
# This returns an internal PySCF molecule object with the new coordinates
opt_mol = mol_opt

# 2. Extract elements and Cartesian coordinates (converted from Bohr back to Angstrom)
# PySCF stores coordinates internally in Bohr, so we convert them using the Bohr constant
from pyscf import __config__
import numpy as np

atoms_count = opt_mol.natm
coords_angstrom = opt_mol.atom_coords() * 0.529177210903  # Bohr to Angstrom conversion

# 3. Construct the XYZ file content
xyz_lines = []
xyz_lines.append(str(atoms_count))
xyz_lines.append(f"Optimized structure | Final Energy: {mf.e_tot:.8f} Hartree")

for i in range(atoms_count):
    symbol = opt_mol.atom_symbol(i)
    x, y, z = coords_angstrom[i]
    xyz_lines.append(f"{symbol:<3} {x:10.5f} {y:10.5f} {z:10.5f}")

# 4. Save directly to your Google Drive
# Adjust the output filename as needed
opt_file_path = "/content/drive/MyDrive/fluoro_Ax_TZV_OPT1.xyz"

with open(opt_file_path, "w") as f:
    f.write("\n".join(xyz_lines))

print(f"🎉 Optimized structure successfully saved to: {opt_file_path}")


In [ ]:
# ========================================================================
# BLOCK 1: EXTRACT AND SAVE OPTIMIZED GEOMETRY TO GOOGLE DRIVE WITH ENERGY
# ========================================================================

# 1. Get the final optimized geometry from the geomeTRIC output object
opt_mol = mol_opt

# NEW STEP: Calculate single-point energy at the final optimized geometry
final_mf = dft.RKS(opt_mol).density_fit()
final_mf.xc = 'cam-b3lyp-d3bj'
final_mf = final_mf.to_gpu()       # Use GPU acceleration
final_energy = final_mf.kernel()   # This computes the exact final energy

# 2. Extract elements and Cartesian coordinates
from pyscf import __config__
import numpy as np

atoms_count = opt_mol.natm
coords_angstrom = opt_mol.atom_coords() * 0.529177210903  # Bohr to Angstrom conversion

# 3. Construct the XYZ file content
xyz_lines = []
xyz_lines.append(str(atoms_count))
# FIX: Using the newly calculated final_energy variable
xyz_lines.append(f"Optimized structure | Final Energy: {final_energy:.8f} Hartree")

for i in range(atoms_count):
    symbol = opt_mol.atom_symbol(i)
    x, y, z = coords_angstrom[i]
    xyz_lines.append(f"{symbol:<3} {x:10.5f} {y:10.5f} {z:10.5f}")

# 4. Save directly to your Google Drive
opt_file_path = "/content/drive/MyDrive/fluoro_pyran_E.xyz"

with open(opt_file_path, "w") as f:
    f.write("\n".join(xyz_lines))

print(f"🎉 Optimized structure successfully saved to: {opt_file_path}")


In [ ]:
# =====================================================================
# CORRECTED BLOCK 2: HESSIAN / FREQUENCY CALCULATION
# =====================================================================
from pyscf import hessian

print("Starting Hessian calculation on the GPU... This may take a few minutes.")

# 1. Re-initialize and execute the DFT engine on your optimized structure (mol_opt)
mf_freq = dft.RKS(mol_opt).density_fit()
mf_freq.xc = 'cam-b3lyp-d3bj'
mf_freq = mf_freq.to_gpu()

# Run the single-point kernel first to converge the wave function
mf_freq.kernel()

# 2. Compute the Hessian matrix on the GPU (Now mf_freq is explicitly defined!)
hess_calc = mf_freq.Hessian()
hess_matrix = hess_calc.kernel()


# 3. Use PySCF's thermo module to convert the Hessian into vibrational frequencies
from pyscf.hessian import thermo
thermo_info = thermo.harmonic_analysis(opt_mol, hess_matrix)

# Extract data using the proper lower-case/standard PySCF keys
frequencies = thermo_info.get('freq_wavenumber', [])

# Fallback mechanism for ZPE extraction depending on PySCF dictionary layout
zpve = thermo_info.get('zpe', thermo_info.get('ZPE', None))

# If neither key exists, calculate it manually from the real frequencies (0.5 * h * nu)
if zpve is None:
    # 1 Hartree = 219474.63 cm^-1, so conversion factor is 0.5 / 219474.63
    real_freqs = [f for f in frequencies if f > 0]
    zpve = sum(real_freqs) * 2.27817e-6

# 4. Print results and check for imaginary frequencies
print("\n" + "="*40)
print("VIBRATIONAL FREQUENCY RESULTS")
print("="*40)
print(f"Zero-Point Vibrational Energy (ZPE): {zpve:.6f} Hartree")
print(f"ZPE in kcal/mol: {zpve * 627.509:.3f} kcal/mol")

# Negative frequencies are returned as real numbers with a negative sign or complex formats
# We check for values significantly less than zero to identify imaginary modes
imaginary_freqs = [f for f in frequencies if f < -1.0]

print(f"\nTotal structural frequencies found: {len(frequencies)}")
if imaginary_freqs:
    print(f"⚠️ Warning: Found {len(imaginary_freqs)} imaginary frequency/frequencies!")
    print(f"Lowest imaginary modes: {imaginary_freqs}")
    print("This structure might be a transition state or a saddle point, not a true minimum.")
else:
    print("✅ Success! Zero imaginary frequencies found. This is a true local minimum structure.")


In [ ]:
# =====================================================================
# INDEPENDENT SPECTRUM EXPORTER
# =====================================================================

# 1. Path for the frequencies file
freq_only_path = "/content/drive/MyDrive/fluoro_Ax_TZV_OPT_Frequencies_Raw.txt"

try:
    # 2. Extract and format into flat data rows
    raw_lines = []
    raw_lines.append("# Mode Number | Frequency (cm^-1)")

    for idx, freq in enumerate(frequencies, 1):
        raw_lines.append(f"{idx:<5} {freq:12.4f}")

    # 3. Write directly out to Drive
    with open(freq_only_path, "w") as f:
        f.write("\n".join(raw_lines))

    print(f"✅ Success! All {len(frequencies)} frequencies saved safely as a flat text file here:")
    print(freq_only_path)

except NameError:
    print("❌ Error: The 'frequencies' variable wasn't found in this cell's memory.")
    print("Please make sure you have run your Block 2 cell successfully first!")


In [ ]:
# =====================================================================
# CORRECTED BLOCK 3: THERMODYNAMIC PROPERTIES
# =====================================================================
from pyscf.hessian import thermo

# 1. Run the thermochemical analysis using the mean-field object and frequencies
thermo_results = thermo.thermo(mf_freq, thermo_info['freq_wavenumber'])

# 2. Helper function to safely extract the numeric value from PySCF (Value, Unit) tuples
def get_val(item):
    if isinstance(item, tuple):
        return item[0]
    return item

print("\n" + "="*40)
print("THERMOCHEMICAL DATA AT 298.15 K, 1 ATM")
print("="*40)
print(f"Total Electronic Energy (E_scf) : {mf_freq.e_tot:.6f} Hartree")
print(f"Zero-Point Energy (ZPE)        : {zpve:.6f} Hartree")

# Extract the standard thermodynamic potentials (unwrapping the tuples)
e_tot = get_val(thermo_results['E_tot'])
h_tot = get_val(thermo_results['H_tot'])
g_tot = get_val(thermo_results['G_tot'])

print(f"Internal Energy (E_tot)        : {e_tot:.6f} au")
print(f"Enthalpy (H)                    : {h_tot:.6f} au")
print(f"Gibbs Free Energy (G)          : {g_tot:.6f} au")





In [ ]:
# =====================================================================
# BLOCK 4: COMPILE DATA AND WRITE TO GOOGLE DRIVE REPORT
# =====================================================================

# Define your output path (change filename as desired)
report_path = "/content/drive/MyDrive/fluoro_Ax_TZV_OPT_Calculation_Report.txt"

# Build the text content structurally
report_content = f"""=====================================================================
QUANTUM CHEMISTRY COMPUTATION REPORT
=====================================================================

1. ELECTRONIC & GEOMETRY ENERGY
---------------------------------------------------------------------
Total Electronic Energy (E_scf) : {mf_freq.e_tot:.8f} Hartree

2. VIBRATIONAL FREQUENCY DATA
---------------------------------------------------------------------
Zero-Point Vibrational Energy (ZPE) : {zpve:.6f} Hartree
ZPE in kcal/mol                     : {zpve * 627.509:.3f} kcal/mol
Total Normal Modes Found            : {len(frequencies)}
Imaginary Frequencies Checked       : {"None (True Local Minimum)" if not [f for f in frequencies if f < -1.0] else "Warning: Imaginary modes present"}

3. THERMOCHEMICAL VALUES (298.15 K, 1 ATM)
---------------------------------------------------------------------
Internal Energy (E_tot)             : {e_tot:.8f} au
Enthalpy (H)                        : {h_tot:.8f} au
Gibbs Free Energy (G)               : {g_tot:.8f} au

=====================================================================
Report generated successfully via GPU-accelerated PySCF environment.
=====================================================================
"""

# Write the text to your drive
with open(report_path, "w") as report_file:
    report_file.write(report_content)

print(f"🎉 Complete summary log written! You can find it at: {report_path}")
print("You can download it directly from your Google Drive sidebar or interface.")


In [ ]:
import pyscf
from pyscf import gto, dft
from pyscf.tools import molden  # Built-in PySCF tool for standard orbital files

# 1. Mount Google Drive if not already done in a previous cell
from google.colab import drive
drive.mount('/content/drive')

# 2. Set the path to your XYZ file sitting on Google Drive
file_path = "/content/drive/MyDrive/fluoro_Ax_TZV_OPT1.xyz"

# 3. Read the file, skipping the first 2 header lines
with open(file_path, "r") as f:
    lines = f.readlines()[2:]
    cartesian_coordinates = "".join(lines)

# 4. Initialize PySCF Molecule with the def2-tzvpd basis set
mol = gto.M(
    atom=cartesian_coordinates,
    basis='def2-tzvpd',
    charge=0,
    spin=0
)

# 5. Build the Density Functional Theory environment on the GPU
mf = dft.RKS(mol).density_fit()
mf.xc = 'cam-b3lyp-d3bj'    # Functional + dispersion settings
mf = mf.to_gpu()            # Push matrices directly onto the GPU

# =====================================================================
# 6. EXECUTE ENERGY CALCULATION & EXPORT NATIVE .molden FILE
# =====================================================================
print("[INFO] Computing GPU-accelerated single-point energy...")
mf.kernel()                 # Solve the SCF equations on the GPU

# Pull the converged state data arrays safely back to standard system memory
cpu_mf = mf.to_cpu()

# Write the complete wavefunctions into a standardized .molden format
molden_filename = "f_Ax_thp_E.molden"
molden.from_scf(cpu_mf, molden_filename) # Built-in, error-proof dumper

print(f"\n[SUCCESS] Single-point energy calculation complete!")
print(f"[SUCCESS] Native .molden file generated as '{molden_filename}' in your Colab folder.")
print("Download this file, drag it onto Multiwfn.exe, and proceed normally.")

In [ ]:
import pyscf, gpu4pyscf, geometric
print(f"PySCF Version:       {pyscf.__version__}")
print(f"GPU4PySCF Version:   {gpu4pyscf.__version__}")
print(f"geomeTRIC Version:   {geometric.__version__}")
